**IMPORTS**

In [1]:
from Configuracion import CARGADOS, PROCESADOS, CSV_GENERADOS, IMPLEMENTACION_GOBERNANZA, ANALITICOS_PARQUET, ANALITICOS_RESULTADOS
from Configuracion import RECURSOS

import os
import re
from collections import Counter

**SESION DE SPARK**

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("CybersecurityDataAnalytics")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/20 01:16:34 WARN Utils: Your hostname, DESKTOP-UPI0SF1, resolves to a loopback address: 127.0.1.1; using 172.31.238.137 instead (on interface eth0)
26/09/20 01:16:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/vctor/Proyecto_GDIABD_Cybersecurity_Data_Analytics/venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/20 01:16:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


**RUTAS**

In [3]:
path_crudo = os.path.join(PROCESADOS, "dataset_crudo_unificado.parquet")
path_ml    = os.path.join(PROCESADOS, "dataset_ml_unificado.parquet")

os.makedirs(ANALITICOS_PARQUET, exist_ok=True)

**CARGAR LOS DATOS**

In [4]:
df_crudo = spark.read.parquet(path_crudo)
df_ml    = spark.read.parquet(path_ml)

print("df_crudo filas:", df_crudo.count(), "| columnas:", len(df_crudo.columns))
print("df_ml filas:", df_ml.count(), "| columnas:", len(df_ml.columns))

df_crudo filas: 3119345 | columnas: 85
df_ml filas: 2830743 | columnas: 79


**NORMALIZACION DE NOMBRES Y DUPLICADOS (ANOMALIA DE COLUMNA DUPLICADA DEL DATASET)**

In [5]:
def normalizar_y_deduplicar(df):
    nombres_limpios = []
    for c in df.columns:
        limpio = c.strip().lower().replace(' ', '_').replace('/', '_').replace('.', '_')
        limpio = re.sub(r'[^a-z0-9_]', '', limpio)
        nombres_limpios.append(limpio)

    contador = Counter()
    nombres_finales = []
    for nombre in nombres_limpios:
        contador[nombre] += 1
        if contador[nombre] == 1:
            nombres_finales.append(nombre)
        else:
            nombres_finales.append(f"{nombre}_dup{contador[nombre]}")

    return df.toDF(*nombres_finales)

df_crudo = normalizar_y_deduplicar(df_crudo)
df_ml    = normalizar_y_deduplicar(df_ml)

print("Columnas df_crudo:", df_crudo.columns)

Columnas df_crudo: ['flow_id', 'source_ip', 'source_port', 'destination_ip', 'destination_port', 'protocol', 'timestamp', 'flow_duration', 'total_fwd_packets', 'total_backward_packets', 'total_length_of_fwd_packets', 'total_length_of_bwd_packets', 'fwd_packet_length_max', 'fwd_packet_length_min', 'fwd_packet_length_mean', 'fwd_packet_length_std', 'bwd_packet_length_max', 'bwd_packet_length_min', 'bwd_packet_length_mean', 'bwd_packet_length_std', 'flow_bytes_s', 'flow_packets_s', 'flow_iat_mean', 'flow_iat_std', 'flow_iat_max', 'flow_iat_min', 'fwd_iat_total', 'fwd_iat_mean', 'fwd_iat_std', 'fwd_iat_max', 'fwd_iat_min', 'bwd_iat_total', 'bwd_iat_mean', 'bwd_iat_std', 'bwd_iat_max', 'bwd_iat_min', 'fwd_psh_flags', 'bwd_psh_flags', 'fwd_urg_flags', 'bwd_urg_flags', 'fwd_header_length', 'bwd_header_length', 'fwd_packets_s', 'bwd_packets_s', 'min_packet_length', 'max_packet_length', 'packet_length_mean', 'packet_length_std', 'packet_length_variance', 'fin_flag_count', 'syn_flag_count', 'rst

**DIAGNOSTICO**

In [6]:
def diagnostico_spark(df, nombre):
    print(f"=== {nombre} ===")
    df.printSchema()

    total = df.count()
    print(f"Filas: {total:,}")

    print("\nNulos por columna (solo columnas con > 0):")
    nulos = df.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns
    ]).collect()[0].asDict()
    nulos_con_valor = {k: v for k, v in nulos.items() if v > 0}
    for col, cantidad in sorted(nulos_con_valor.items(), key=lambda x: -x[1]):
        print(f"  {col}: {cantidad:,}")

    duplicados = total - df.dropDuplicates().count()
    print(f"\nFilas duplicadas: {duplicados:,}")
    print("="*50)

diagnostico_spark(df_crudo, "df_crudo")
diagnostico_spark(df_ml, "df_ml")

=== df_crudo ===
root
 |-- flow_id: string (nullable = true)
 |-- source_ip: string (nullable = true)
 |-- source_port: double (nullable = true)
 |-- destination_ip: string (nullable = true)
 |-- destination_port: double (nullable = true)
 |-- protocol: double (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- flow_duration: double (nullable = true)
 |-- total_fwd_packets: double (nullable = true)
 |-- total_backward_packets: double (nullable = true)
 |-- total_length_of_fwd_packets: double (nullable = true)
 |-- total_length_of_bwd_packets: double (nullable = true)
 |-- fwd_packet_length_max: double (nullable = true)
 |-- fwd_packet_length_min: double (nullable = true)
 |-- fwd_packet_length_mean: double (nullable = true)
 |-- fwd_packet_length_std: double (nullable = true)
 |-- bwd_packet_length_max: double (nullable = true)
 |-- bwd_packet_length_min: double (nullable = true)
 |-- bwd_packet_length_mean: double (nullable = true)
 |-- bwd_packet_length_std: double (nulla

26/09/20 01:17:09 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


  flow_bytes_s: 289,960
  flow_id: 288,602
  source_ip: 288,602
  source_port: 288,602
  destination_ip: 288,602
  destination_port: 288,602
  protocol: 288,602
  timestamp: 288,602
  flow_duration: 288,602
  total_fwd_packets: 288,602
  total_backward_packets: 288,602
  total_length_of_fwd_packets: 288,602
  total_length_of_bwd_packets: 288,602
  fwd_packet_length_max: 288,602
  fwd_packet_length_min: 288,602
  fwd_packet_length_mean: 288,602
  fwd_packet_length_std: 288,602
  bwd_packet_length_max: 288,602
  bwd_packet_length_min: 288,602
  bwd_packet_length_mean: 288,602
  bwd_packet_length_std: 288,602
  flow_packets_s: 288,602
  flow_iat_mean: 288,602
  flow_iat_std: 288,602
  flow_iat_max: 288,602
  flow_iat_min: 288,602
  fwd_iat_total: 288,602
  fwd_iat_mean: 288,602
  fwd_iat_std: 288,602
  fwd_iat_max: 288,602
  fwd_iat_min: 288,602
  bwd_iat_total: 288,602
  bwd_iat_mean: 288,602
  bwd_iat_std: 288,602
  bwd_iat_max: 288,602
  bwd_iat_min: 288,602
  fwd_psh_flags: 288,602
  


Filas duplicadas: 288,804
=== df_ml ===
root
 |-- destination_port: long (nullable = true)
 |-- flow_duration: long (nullable = true)
 |-- total_fwd_packets: long (nullable = true)
 |-- total_backward_packets: long (nullable = true)
 |-- total_length_of_fwd_packets: long (nullable = true)
 |-- total_length_of_bwd_packets: long (nullable = true)
 |-- fwd_packet_length_max: long (nullable = true)
 |-- fwd_packet_length_min: long (nullable = true)
 |-- fwd_packet_length_mean: double (nullable = true)
 |-- fwd_packet_length_std: double (nullable = true)
 |-- bwd_packet_length_max: long (nullable = true)
 |-- bwd_packet_length_min: long (nullable = true)
 |-- bwd_packet_length_mean: double (nullable = true)
 |-- bwd_packet_length_std: double (nullable = true)
 |-- flow_bytes_s: double (nullable = true)
 |-- flow_packets_s: double (nullable = true)
 |-- flow_iat_mean: double (nullable = true)
 |-- flow_iat_std: double (nullable = true)
 |-- flow_iat_max: long (nullable = true)
 |-- flow_iat

  flow_bytes_s: 1,358



Filas duplicadas: 308,381


**VERIFICAR SI LAS COLUMNAS DUPLICADAS SON EXACTAMENTE IGUALES**

In [7]:
print("Solo fwd_header_length:", df_crudo.select("fwd_header_length").distinct().count())
print("Combinación de ambas:", df_crudo.select("fwd_header_length", "fwd_header_length_1").distinct().count())

Solo fwd_header_length: 3772
Combinación de ambas: 3772


**ELIMINAR DUPLICADA**

In [8]:
df_crudo = df_crudo.drop("fwd_header_length_1")
df_ml    = df_ml.drop("fwd_header_length_1")

print("Columnas restantes en df_crudo:", len(df_crudo.columns))
print("Columnas restantes en df_ml:", len(df_ml.columns))

Columnas restantes en df_crudo: 84
Columnas restantes en df_ml: 78


**NORMALIZACION DE VALORES DE TEXTO**

In [9]:
def normalizar_valores_texto_spark(df):
    columnas_texto = [c for c, tipo in df.dtypes if tipo == 'string']
    print("Columnas de texto detectadas:", columnas_texto)
    for c in columnas_texto:
        df = df.withColumn(c, F.trim(F.col(c)))
    return df

df_crudo = normalizar_valores_texto_spark(df_crudo)
df_ml    = normalizar_valores_texto_spark(df_ml)

Columnas de texto detectadas: ['flow_id', 'source_ip', 'destination_ip', 'timestamp', 'label']
Columnas de texto detectadas: ['label']


**REVISAR VALORES UNICOS**

In [10]:
print(f"=== label en df_crudo ===")

# Cuántos nulos hay en label
nulos_label = df_crudo.filter(F.col("label").isNull()).count()
print(f"Filas con label nulo: {nulos_label:,}")

# Valores no nulos, ordenados
valores_crudo = [r[0] for r in df_crudo.select("label").distinct().collect() if r[0] is not None]
print(sorted(valores_crudo))

=== label en df_crudo ===
Filas con label nulo: 288,602
['BENIGN', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'Heartbleed', 'Infiltration', 'PortScan', 'SSH-Patator', 'Web Attack \x96 Brute Force', 'Web Attack \x96 Sql Injection', 'Web Attack \x96 XSS']


In [11]:
# ¿Las filas sin label vienen de un archivo/fuente específica?
df_crudo.filter(F.col("label").isNull()).groupBy("source_file").count().show() if "source_file" in df_crudo.columns else print("No hay columna source_file para rastrear el origen")

No hay columna source_file para rastrear el origen


In [12]:
print("Nulos en label (df_ml):", df_ml.filter(F.col("label").isNull()).count())

Nulos en label (df_ml): 0


In [13]:
df_crudo = df_crudo.withColumn("label", F.regexp_replace(F.col("label"), "\x96", "-"))
df_ml    = df_ml.withColumn("label", F.regexp_replace(F.col("label"), "\x96", "-"))

# Verificar
valores_corregidos = [r[0] for r in df_crudo.select("label").distinct().collect() if r[0] is not None]
print(sorted(valores_corregidos))

['BENIGN', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'Heartbleed', 'Infiltration', 'PortScan', 'SSH-Patator', 'Web Attack - Brute Force', 'Web Attack - Sql Injection', 'Web Attack - XSS']


**ELIMINAR FILAS NULAS DE DF_CRUDO**

In [14]:
antes = df_crudo.count()
df_crudo = df_crudo.filter(F.col("label").isNotNull())
print(f"df_crudo: {antes:,} -> {df_crudo.count():,} filas ({antes - df_crudo.count():,} filas sin etiqueta eliminadas)")

df_crudo: 3,119,345 -> 2,830,743 filas (288,602 filas sin etiqueta eliminadas)


**DUPLICADOS DE FILA**

In [15]:
antes = df_ml.count()
df_ml = df_ml.dropDuplicates()
print(f"df_ml: {antes:,} -> {df_ml.count():,} filas")

antes = df_crudo.count()
df_crudo = df_crudo.dropDuplicates()
print(f"df_crudo: {antes:,} -> {df_crudo.count():,} filas")

df_ml: 2,830,743 -> 2,522,362 filas


df_crudo: 2,830,743 -> 2,830,540 filas


In [16]:
# Distribución de label ANTES de guardar
df_ml.groupBy("label").count().orderBy(F.desc("count")).show(20, truncate=False)

+----------------------------+-------+
|label                       |count  |
+----------------------------+-------+
|BENIGN                      |2096484|
|DoS Hulk                    |172849 |
|DDoS                        |128016 |
|PortScan                    |90819  |
|DoS GoldenEye               |10286  |
|FTP-Patator                 |5933   |
|DoS slowloris               |5385   |
|DoS Slowhttptest            |5228   |
|SSH-Patator                 |3219   |
|Bot                         |1953   |
|Web Attack ï¿½ Brute Force  |1470   |
|Web Attack ï¿½ XSS          |652    |
|Infiltration                |36     |
|Web Attack ï¿½ Sql Injection|21     |
|Heartbleed                  |11     |
+----------------------------+-------+



**GUARDAR**

In [ ]:
df_crudo.write.mode("overwrite").parquet(os.path.join(ANALITICOS_PARQUET, "dataset_crudo_limpio_spark"))
df_ml.write.mode("overwrite").parquet(os.path.join(ANALITICOS_PARQUET, "dataset_ml_limpio_spark"))

print("Guardado en:", ANALITICOS_PARQUET)

spark.stop()

**REVERIFICACION DE NULOS**

In [ ]:
nulos_ml = df_ml.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_ml.columns
]).collect()[0].asDict()
nulos_ml_con_valor = {k: v for k, v in nulos_ml.items() if v > 0}
print("Nulos restantes en df_ml:")
for col, cantidad in sorted(nulos_ml_con_valor.items(), key=lambda x: -x[1]):
    print(f"  {col}: {cantidad:,}")

nulos_crudo = df_crudo.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_crudo.columns
]).collect()[0].asDict()
nulos_crudo_con_valor = {k: v for k, v in nulos_crudo.items() if v > 0}
print("\nNulos restantes en df_crudo:")
for col, cantidad in sorted(nulos_crudo_con_valor.items(), key=lambda x: -x[1]):
    print(f"  {col}: {cantidad:,}")

Nulos restantes en df_ml:
  flow_bytes_s: 353
